In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2024-12-18 15:41:21--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  1.48MB/s    in 0.7s    

2024-12-18 15:41:23 (1.48 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

## Step 1 - Tokenisation (Encoder)

In [6]:
chars = sorted(list(set(text))) # all possible characters

# Create a dictionary to map characters to integers (string to int)
stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i # assigning an int value 

# Create a dictionary to map integers to characters (int to string)
itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch # assigning a string value

# Function to encode a string (str -> list of int)
def encode(s):
    result = []
    for c in s:
        result.append(stoi[c])  # Convert each character to its corresponding integer
    return result

# Function to decode a list of integers (list of int -> str)
def decode(l):
    result = ""
    for i in l:
        result += itos[i]  # Convert each integer to its corresponding character
    return result

print(encode("Yello!"))

[37, 43, 50, 50, 53, 2]


## Step 2 - Generating data for Training

In [15]:
# Encoding Tiny shake and store as a torch.tensor
import torch # PyTorch framework 

tinyshake = torch.tensor(encode(text), dtype=torch.long)
print(tinyshake.shape, tinyshake.dtype)

torch.Size([1115394]) torch.int64


### Splitting Train & Validation Datasets

In [16]:
# splitting train & validation datasets
n = int(0.9*len(tinyshake))
train_data = tinyshake[:n]
val_data = tinyshake[n:]

print("Train Length: " + str(len(train_data)))
print("Val Length: " + str(len(val_data)))

Train Length: 1003854
Val Length: 111540


### Using "**Blocks**" To enhance training in parallel

In [19]:
blockSize = 14 # maximum context window

x = train_data[:blockSize]
y = train_data[:blockSize + 1]
for t in range(blockSize):
    context = x[:t+1]
    target = y[t]
    print(f"When input (x) is {str(context)}, target/output (y) is {str(target)}")

When input (x) is tensor([18]), target/output (y) is tensor(18)
When input (x) is tensor([18, 47]), target/output (y) is tensor(47)
When input (x) is tensor([18, 47, 56]), target/output (y) is tensor(56)
When input (x) is tensor([18, 47, 56, 57]), target/output (y) is tensor(57)
When input (x) is tensor([18, 47, 56, 57, 58]), target/output (y) is tensor(58)
When input (x) is tensor([18, 47, 56, 57, 58,  1]), target/output (y) is tensor(1)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15]), target/output (y) is tensor(15)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target/output (y) is tensor(47)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]), target/output (y) is tensor(58)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]), target/output (y) is tensor(47)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64]), target/output (y) is tensor(64)
When input (x) is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43]), t

In [21]:
torch.manual_seed(1337)
batch_size = 4 # independent sequences processed in PARALLEL

def get_batch(split):
    # Getting data
    if split == 'train':
        data = train_data
    else:
        data = val_data

    ix = torch.randint(len(data) - blockSize, (batch_size,))

    # stacking "chunks" of data
    x = []
    y = []
    for i in ix:
        x.append(data[i:i+blockSize])
        y.append(data[i+1:i+blockSize+1])
    x = torch.stack(x)
    y = torch.stack(y)

    return x, y

## Step 3 - Defining the Model (Bigram)

In [22]:
class Bigram(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding__table = torch.nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets):
        logits = self.token_embedding__table(idx)
        return logits

torch.Size([4, 8, 2])